# K-mer Feature Pipeline (Counts)

This notebook repeats the selection & modeling flow but uses actual k-mer counts
for the final modeling step (with `log1p` scaling). Selection still uses binary
presence/absence for robustness.

In [ ]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_selection import chi2
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.preprocessing import StandardScaler
import joblib

## Helpers
Reuse the same `iter_genome_kmers`, `build_prevalence`, and selection helpers as the
binary notebook. The final matrix will be built with counts and transformed.

In [ ]:
# Helper utilities for k-mer pipelines (counts notebook)
# Each function includes a short docstring explaining purpose, parameters, and return value.
def iter_genome_kmers(dump_path: Path) -> dict[str, int]:
    """Read a per-genome k-mer dump and return a dict of {kmer: count}.

    Parameters:
    - dump_path: Path to a single genome's k-mer dump file (two columns: kmer count).

    Returns:
    - dict mapping kmer (str) to integer count.
    """
    kmers: dict[str, int] = {}
    with dump_path.open('r', encoding='utf8', errors='ignore') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 2:
                continue
            kmer, cnt = parts
            try:
                kmers[kmer] = int(cnt)
            except ValueError:
                continue
    return kmers

def build_prevalence(dump_dir: str, genome_ids: list[str]) -> Counter:
    """Aggregate presence counts for each k-mer across a list of genomes.

    Scans each genome's k-mer dump and counts how many genomes contain each k-mer (presence, not counts).

    Parameters:
    - dump_dir: directory containing `{GenomeID}_db_kmers.txt` files.
    - genome_ids: list of genome identifiers to include.

    Returns:
    - Counter where keys are k-mers and values are the number of genomes containing that k-mer.
    """
    prevalence = Counter()
    dump_dir = Path(dump_dir)
    for gid in genome_ids:
        dump_path = dump_dir / f'{gid}_db_kmers.txt'
        if not dump_path.exists():
            continue
        kmers = iter_genome_kmers(dump_path)
        # update counts by presence (keys only)
        prevalence.update(kmers.keys())
    return prevalence

def select_vocab_by_prevalence(prevalence: Counter, n_genomes: int,
                                   min_frac: float = 0.01, max_frac: float = 0.99) -> list[str]:
    """Filter k-mers by prevalence across genomes to remove extremely rare or ubiquitous features.

    Parameters:
    - prevalence: Counter from `build_prevalence` mapping k-mer -> genome occurrence count.
    - n_genomes: number of genomes used to compute prevalence (for fraction -> absolute conversion).
    - min_frac: minimum fraction of genomes that must contain a k-mer to keep it.
    - max_frac: maximum fraction of genomes that may contain a k-mer to keep it.

    Returns:
    - List of k-mer strings that pass the prevalence filters.
    """
    min_count = int(np.ceil(min_frac * n_genomes))
    max_count = int(np.floor(max_frac * n_genomes))
    vocab = [k for k, c in prevalence.items() if min_count <= c <= max_count]
    return vocab

def build_sparse_matrix(dump_dir: str, genome_ids: list[str], vocab: list[str], binary: bool = False) -> sparse.csr_matrix:
    """Construct a sparse (CSR) matrix of shape (n_genomes, n_features) from k-mer dumps.

    Reads each genome's dump and populates the matrix using the provided `vocab` index.
    When `binary` is True, presence is recorded as 1; otherwise counts are used (int).

    Parameters:
    - dump_dir: directory with per-genome k-mer dump files.
    - genome_ids: ordered list of genome IDs corresponding to rows in the matrix.
    - vocab: ordered list of k-mers corresponding to columns in the matrix.
    - binary: whether to collapse counts to binary presence/absence.

    Returns:
    - scipy.sparse.csr_matrix with dtype `np.int32` for counts (or `np.int8` for binary).
    """
    vocab_index = {kmer: idx for idx, kmer in enumerate(vocab)}
    rows = []
    cols = []
    data = []
    dump_dir = Path(dump_dir)
    for row_idx, gid in enumerate(genome_ids):
        dump_path = dump_dir / f'{gid}_db_kmers.txt'
        if not dump_path.exists():
            continue
        kmers = iter_genome_kmers(dump_path)
        for kmer, cnt in kmers.items():
            col_idx = vocab_index.get(kmer)
            if col_idx is None:
                continue
            rows.append(row_idx)
            cols.append(col_idx)
            data.append(1 if binary else int(cnt))
    dtype = np.int8 if binary else np.int32
    mat = sparse.csr_matrix((data, (rows, cols)), shape=(len(genome_ids), len(vocab)), dtype=dtype)
    return mat

def load_labels(labels_path: Path, treat_intermediate_as_resistant: bool = True) -> pd.Series:
    """Load phenotype labels from a CSV and return a binary series indexed by GenomeID.

    Expects a CSV with at least `GenomeID` and `phenotype` columns. Optionally maps 'I' -> 'R'.

    Parameters:
    - labels_path: Path to CSV containing labels.
    - treat_intermediate_as_resistant: if True, map 'I' to 'R' before filtering.

    Returns:
    - pandas Series indexed by GenomeID with values 1 for resistant and 0 for susceptible.
    """
    df = pd.read_csv(labels_path)
    if 'GenomeID' not in df.columns or 'phenotype' not in df.columns:
        raise ValueError('labels CSV must contain GenomeID and phenotype columns')
    pheno = df.set_index('GenomeID')['phenotype'].str.upper()
    if treat_intermediate_as_resistant:
        pheno = pheno.replace({'I': 'R'})
    pheno = pheno[pheno.isin(['R', 'S'])]
    y = pheno.map({'R': 1, 'S': 0})
    return y

In [ ]:
# ---- Run counts-based flow (edit paths before running) ----
dump_dir = Path('output/counted_kmers')
labels_path = Path('data/isolates.csv')  # expects columns: GenomeID, phenotype (R/I/S)
genome_ids_path = Path('data/genome_ids.txt')

# Load genome ids
genome_ids = []
if genome_ids_path.exists():
    with genome_ids_path.open('r') as f:
        genome_ids = [line.strip() for line in f if line.strip()]

# Use the `load_labels` helper defined in the helpers cell
y = load_labels(labels_path, treat_intermediate_as_resistant=True)
genome_ids = [gid for gid in genome_ids if gid in y.index]
print(f'Using {len(genome_ids)} genomes for selection')

# 1) Prevalence counting
prevalence = build_prevalence(dump_dir, genome_ids)
print(f'Unique k-mers observed: {len(prevalence):,d}')

# 2) Prevalence filter
vocab = select_vocab_by_prevalence(prevalence, n_genomes=len(genome_ids), min_frac=0.01, max_frac=0.99)
print(f'Vocab after prevalence filter: {len(vocab):,d}')

# 3) Build binary sparse matrix for selection
X_bin = build_sparse_matrix(dump_dir, genome_ids, vocab, binary=True)
y_arr = y.loc[genome_ids].to_numpy()
print('Built binary matrix for selection:', X_bin.shape)

# 4) Chi-square selection
top_k = min(100000, X_bin.shape[1])
from sklearn.feature_selection import chi2
scores, _ = chi2(X_bin, y_arr)
top_idx = np.argsort(scores)[::-1][:top_k]
vocab_sel = [vocab[i] for i in top_idx]
print(f'Selected top k-mers: {len(vocab_sel):,d}')

# 5) Build final counts matrix for modeling (only selected features)
X_counts = build_sparse_matrix(dump_dir, genome_ids, vocab_sel, binary=False)
print('Built counts matrix:', X_counts.shape)

# 6) Transform counts (log1p) and scale if needed
Xc = X_counts.astype(np.float32)
Xc.data = np.log1p(Xc.data)
scaler = StandardScaler(with_mean=False)
Xc_scaled = scaler.fit_transform(Xc)

# 7) Train and evaluate
X_train, X_test, y_train, y_test = train_test_split(Xc_scaled, y_arr, test_size=0.2, random_state=42, stratify=y_arr)
log = LogisticRegression(max_iter=1000, solver='saga')
log.fit(X_train, y_train)
y_pred = log.predict(X_test)
print('Logistic accuracy (counts):', accuracy_score(y_test, y_pred))

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print('RF accuracy (counts):', accuracy_score(y_test, y_pred_rf))

# 8) Save artifacts
out_dir = Path('output/feature_selection_counts')
out_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(vocab_sel, out_dir / 'vocab_selected_counts.pkl')
joblib.dump(log, out_dir / 'logistic_counts.joblib')
joblib.dump(rf, out_dir / 'rf_counts.joblib')
with open(out_dir / 'results_counts.json', 'w') as fh:
    json.dump({'logistic_acc': accuracy_score(y_test, y_pred), 'rf_acc': accuracy_score(y_test, y_pred_rf)}, fh, indent=2)
print('Saved selected vocabulary and models to', out_dir)